# CRUD et formulaires


In [ ]:
# on restaure notre base de données en remplaçant la BDD par son backup 
!cp ./richelieu.db.bak ./richelieu.db

On sait maintenant:
- créer une appli Flask
- la connecter à une base de données via SQLAlchemy  
- modéliser une BDD via l'ORM SQLAlchemy
- parcourir les objets de l'ORM, et leurs relations
- faire des requêtes `SELECT` avec SQLAlchemy
- créer des templates Jinja

**En bref**: tout pour faire un site "catalogue", qui affiche des données sans pouvoir .

Aujourd'hui, on va voir comment faire **toutes les opérations CRUD** sur notre BDD directement depuis le site. On va donc apprendre à:
- **faire des requêtes `CREATE`, `UPDATE` et `DELETE`** avec SQLAlchemy (on sait déjà faire le `READ`, on saura donc faire toutes les opérations CRUD) 
- **créer des formulaires HTML** pour ajouter/modifier/supprimer des données depuis le site
- **gérer des comptes d'utilisateur.ice.s**

---

# Apparté bonnes pratiques

On code maintenant ensemble depuis 8h et on va commencer à voir des fonctions un peu plus longues.

Rappelez vous que **vous codez pour le long terme**: votre code devra être utilisé par vous et par d'autres, pendant plusieurs années. Retenez que c'est **plus dur de lire du code que d'en écrire** (il faut s'adapter à une autre logique, traduire le code en langage humain...), et c'est **beaucoup plus dur de lire du code de quelqu'un d'autre** (il faut s'adapter à la logique de quelqu'un d'autre). **Pour coder, il faut avoir beaucoup d'informations en tête** (noms de variables, ce qu'elles contiennent, ce qu'on veut en faire, comment...). 

Votre but, c'est d'être **le plus descriptif pour réduire le volume d'informations que vous devez retenir** dans votre "RAM mentale".

**Voici donc quelques bonnes pratiques à adopter**:
- **commentez votre code**: chaque fonction doit avec une docstring
- **une fonction = une opération**: une fonction doit faire une chose, et le faire bien. Plus une fonction est longue, 
    - plus elle est complexe 
    - dure à comprendre
    - plus il y a un risque d'erreur
- **faites des fonctions courtes**: une fonction doit faire au grand maximum la hauteur de votre écran (format paysage)  
- **soyez explicites** dans nos noms de variables, de fonctions et de classes:
    - une variable appelée `x` ou une fonction `f`, ça ne dit pas grand chose sur ce que contient la variable, ou ce que la fonciton fait
    - **votre objectif**:
        - vos variables décrivent leur contenu
        - vos noms de fonction décrivent l'opération faite par la fonction 
        - écrire du code synthétique, c'est moins important qu'écrire du code compréhensible !
- **typez vos fonctions**: peut-être le plus important. **Typer, c'est expliciter ce que contient une variable, et ce que fait une fonction**. C'est **très douloureux** de devoir mettre des prints partout pour comprendre ce que contiennent des variables, et les type hints permettent d'alléger beaucoup ça.

En bref, 
```py
# cette fonction
def f(a, b):
    if len(a) > b:
        return f"{a[:b]}[...]"
    return a

# est beaucoup moins claire que:
def raccourcir(string, max_len):
    if len(string) > max_len:
        return f"{string[:max_len]}[...]"
    return string

# qui est moins claire que:
def raccourcir_avec_types(string: str, max_len: int) -> str:
    if len(string) > max_len:
        return f"{string[:max_len]}[...]"
    return string
```



---

# Sécuriser une application

Avant de commencer à créer des comptes dans tous les sens, petit point sécurité. À partir du moment où on permet à des utilisateur.ice.s de créer leur compte, 
- on leur permet de faire des CREATE/UPDATE/DELETE sur la BDD pour modifier leur compte
- on leur stocke aussi leurs données, et on est responsable de données sensibles.

Il y a **énormément** de fuites de données et d'attaques informatiques, surtout depuis les LLM et surtout envers les institutions de recherche et de conservation. Par exemple, en 2023, une [cyberattaque sur la British Library](https://en.wikipedia.org/wiki/British_Library_cyberattack) bloque l'accès à ses collections en ligne (entre beaucoup de choses).

On va donc voir comment bien gérer tout ça.

## Sécurisation de mots de passe: le hashage

### La théorie

⚠️⚠️⚠️ **ON NE STOCKE JAMAIS UN MOT DE PASSE EN BRUT DANS UNE BASE DE DONNÉES**. Quand on créée un compte en ligne, ou un compte d'utilisateur sur son ordinateur, **c'est un hash du mot de passe qui est stocké**. 

Un **hash, c'est une chaîne de caractères** générée par une "fonction de hashage" à partir d'une chaîne de caractères en entrée. Ce `hash` est 
- **irréversible**: si `tartempion` est hashé en `xff3eoa42`, il est impossible de retrouver `tartempion` à partir de `xff3eoa42`.
- **unique** à une valeur d'entrée: le hash `xff3eoa42` ne peut être obtenu qu'avec l'entrée `tartempion`.

Par exemple, votre ordinateur ne sait pas quel est votre mot de passe: quand vous vous connectez, il calcule le hash de votre mot de passe et vérifie si cela correspond au hash qu'il a enregistré. Pour les mots de passe d'une application, c'est pareil !

(Mais par contre, il arrive qu'un algorithme de hashage soit "cracké": on peut trouver une manière d'obtenir le mot de passe à partir de son hash. Dans ce cas, il s'agit d'une grosse faille de sécurité.)

#### La pratique: hasher un mot de passe

Écrire un algo de hashage dépasse très largement mes compétences. Fort heureusement, **Werkzeug (librairie sous-couche de Flask) a deux fonctions qui le hashage** ce problème pour nous:

- `generate_password_hash()`: créer un hash à partir d'un mot de passe (utilisé pour définir un mot de passe)
- `check_password_hash()`: vérifier que le mot de passe fourni est correct (utilisé quand on se connecte) 

In [ ]:
from werkzeug.security import generate_password_hash, check_password_hash

mdp = "tartempion"

mdp_hash = generate_password_hash(mdp)
print("le hash est:", mdp_hash)

print("check_password_hash avec le bon mdp:", check_password_hash(mdp_hash, mdp))
print("check_password_hash avec le mauvais mdp:", check_password_hash(mdp_hash, "ceci est une vilaine tentative d'intrusion"))

## Sécurisation de l'application: la `SECRET_KEY`

En interne, Flask doit confirmer l'authenticité de plein de choses (par exemple, le cookie qui permet à un.e utilisateur.ice de rester connecté.e d'une page à l'autre). Pour cela, Flask a besoin que l'on définisse une `SECRET_KEY`.

**On définit `SECRET_KEY`** dans `app/utils/constants.py`:
```py
from warnings import warn

# la clé top secrète
secret_key_default = "Une clé secrète"
SECRET_KEY = "Une clé secrète"

if SECRET_KEY == secret_key_default:
    warn(f"Changez votre clé secrète avant de passer en production ! Clé secrète actuelle: {SECRET_KEY}")
```

Et dans `app/app.py`, **on ajoute `SECRET_KEY` à la configuration de notre appli** Flask:

```py
app = Flask(
    APP_NAME,
    template_folder=DIR_TEMPLATES, 
    static_folder=DIR_STATICS
)
app.config["SQLALCHEMY_DATABASE_URI"] = f"sqlite:///{PATH_DB}"
app.config["SECRET_KEY"] = SECRET_KEY
db = SQLAlchemy(app)
```

**À noter**: une `SECRET_KEY` doit toujours être confidentielle. C'est pourquoi dans `constants.py`, on voit un `warn()`. **Avant de mettre une appli en production, il faut toujours définir une clé secrète *random*** via un algorithme solide.

> **Lancez l'appli `apps/s5/secret_key` et regardez le warning qui s'affiche (le reste de l'appli n'a pas changé)**
> ```py
> python apps/s5/secret_key/main.py
> ```


---

# Les `users` et première requête `CREATE`

![db schema](./img/db_schema.png)

Votre regard aguisé aura remarqué que pour le moment, on a pas modélisé les tables `user` et `user_iconography`. Elles servent à:
- ajouter de la gestion d'utilisateur.ice.s sur le site
- tracer qui modifie la table `Iconography`.

## `User`

**Voici notre table `User`** de base. Chaque ligne de la table correspond à un.e utilisateur.ice différent.e.

```py
class User(db.Model):
    __tablename__ = "user"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    user_name: Mapped[str]
    user_mail: Mapped[str]
    user_password: Mapped[str]
    # NOTE: la relation à `Iconography` sera définie plus bas
```

## Première requête `CREATE`

Je fais un exemple très simple d'insertion SQL dans la table `User`. Pour simplifier, mon modèle pour `User` ne contient pas la relation avec `Iconography`.

In [ ]:
from typing import List, Optional
from pathlib import Path

from flask import Flask
from flask_sqlalchemy import SQLAlchemy
from werkzeug.security import generate_password_hash
from sqlalchemy import ForeignKey
from sqlalchemy.orm import Mapped, mapped_column, relationship

path_to_db = Path("./richelieu.db").absolute()
print(path_to_db)

APP_NAME = "Catalogue Richelieu"
app = Flask(APP_NAME)
app.config["SQLALCHEMY_DATABASE_URI"] = f"sqlite:///{path_to_db}"
db = SQLAlchemy(app)

class User(db.Model):
    __tablename__ = "user"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    user_name: Mapped[str]
    user_mail: Mapped[str]
    user_password: Mapped[str]
    # NOTE: la relation à `Iconography` sera définie plus bas

### Créer un `User` et le sauvegarder

#### 1. Créer un user côté Python

Pour créer un `User`, rien de plus facile. Nos modèles de base de données sont des classes, donc `User` est une classe. Donc,chaque utilisateur.ice est représenté par une instance de la classe `User`.

Donc, **créer un nouveau `User`, c'est juste créer un nouvel objet membre de la classe `User`**:

In [ ]:
new_user = User(
    user_name="Nadine Hurley",
    user_mail="nadine.hurley@gmail.com",
    user_password=generate_password_hash("un beau mot de passe bien sécurisé")
) 
print(new_user)
print(new_user.user_name)
print(new_user.user_password)

**Explication de code**: 
- `User()` permet de créer une instance de classe
- les arguments qu'on passe à `User()` sont les valeurs qu'on veut donner à cette instance.

#### 2. Le sauvegarder en base

Pour le moment, `new_user` n'a pas été sauvegardé. En fait, aucune interaction avec la base de données n'a encore eu lieu.

**Il faut faire un `commit` dans la base de données** pour enregistrer `new_user`:

In [ ]:
# pour rappel, on doit utiliser app.app_context dans les notebooks parce qu'on est hors d'une application Flask. 
with app.app_context():
    db.session.add(new_user)
    db.session.commit()
# on remarque maintenant que `new_user` a un ID. 
print(new_user)

**Explication de code**:
- `db.session.add()`: on ajoute `new_user` à la session pour qu'il soit sauvegardé
- `db.session.commit()` sauvegarde tous les changements réalisés pendant une session (create/update/delete).

Le `commit`, si vous vous rappelez, est à la base du SQL: toutes les interactions avec une base de données ont lieu dans une **transaction**, qui est composée de une ou plusieurs requêtes SQL et qui se termine par un **commit**, qui sauvegarde tous les changements de base de données qui ont eu lieu pendant la transaction.

### 3. Une fonction pour créer des cpmptes d'utilisateur.ice.s

À partir de cet exemple minimal, créons une fonction qui prend en paramètres `user_name`, `user_mail` et `user_password` et créé l'`user` si il n'existe pas déjà.

**On ajoute à notre exemple d'au dessus quelques vérifications** pour le rendre plus robuste:

In [ ]:
from typing import Tuple, Union

def create_user(user_mail: str, user_name: str, user_password: str) -> Tuple[bool, Union["User", str]]:
    """
    créer un nouveau user.

    notre fonction retourne:
    - False, <message d'erreur> en cas d'erreur
    - True, User en cas de succès 
    donc, le 1er item permet de savoir si l'insertion a fonctionné 
    """
    # liste de nos erreurs
    errors = []
    
    # 1. on vérifie que l'utilisateur.ice a fourni toutes les données
    if user_mail is None:
        errors.append("Veuillez fournir un email")
    if user_name is None:
        errors.append("Veuillez fournir un nom d'utilisateur")
    if user_password is None:
        errors.append("Veuillez fournir un mot de passe")
    
    # on vérifie si il existe un autre `user` avec le même mail
    existing_user = db.session.execute(
        db.select(User).filter(User.user_mail == user_mail)
    ).scalars().all()
    if len(existing_user):
        errors.append(f"Un utilisateur existe déjà pour le mail: {user_mail}")

    # il y a eu des erreurs => pas d'insert
    if len(errors):
        return False, errors

    # pas d'erreurs => on fait l'insert
    new_user = User(
        user_name=user_name,
        user_mail=user_mail,
        user_password=generate_password_hash(user_password)
    ) 
    try:
        db.session.add(new_user)
        db.session.commit()
    except Exception as e:
        # en cas d'erreur au moment de l'insert, on retourne False et le message d'erreur de l'appli
        print(e)
        return False, str(e)

**Ce qu'on a ajouté**:
- **des vérifications**:
    - une vérification de l'*user input*
    - une vérification pour savoir si un compte existe déjà avec ce mail
- **un `try...except` autour du `commit()`**: la base de données SQL a son propre système de vérifications, et donc des problèmes inattendus peuvent émerger au moment de l'insertion, donc on gère proprement cette erreur. 

## Login: se connecter

Pour se connecter à un compte existant, **l'utilisateur.ice fournit un mail et un mot de passe, et notre application vérifie** si ils correspondent à ce qui est dans la base de données.

**Une fonction pour se connecter** est donc assez facile: 
- l'utilisateur.ice fournit son mail et son mot de passe
- on hashe le mot de passe
- on vérifie qu'il y a une ligne dans la base de données avec le bon mail et le bon hash de mot de passe.
- si oui, on retourne le `user`, sinon on retourne `None`.

In [ ]:
def login(user_mail: str, user_password: str) -> Optional["User"]:
    # retourne soit un `User`, soit None 
    user = db.session.execute(
        db.select(User).filter(User.user_mail == user_mail)
    ).scalars().first()
    # on vérifie que le `User` avec ce mail a bien le bon mot de passe
    if user and check_passoword_hash(user.user_password, user_password):
        return user
    # sinon, on retourne None
    return None

## Tout combiner: le modèle `User` complet

On a vu:
- comment créer un `User` et le sauvegarder en base
- comment se connecter à un compte (c'est-a-dire comment sélectionner le bon `User` en fonction des paramètres fournis par un client)

**On combine tout** dans un modèle `User` définitif. Pour mieux organiser le code, **`create_user` et `login` sont des `staticmethod`** de la classe `User`
- une `staticmethod` est une méthodes de `User` qui n'hérite pas de `self`. Sa présence dans la classe est donc juste une question d'organisation de code.
- pour les utiliser, on fait: `user = User.create_user(...)` et `user = User.login(...)`.

```py
class User(db.Model):
    __tablename__ = "user"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    user_name: Mapped[str]
    user_mail: Mapped[str]
    user_password: Mapped[str]
    
    iconography: Mapped[List["Iconography"]] = relationship(
        back_populates="user",
        secondary=IconographyUser.__table__
    )  

    @staticmethod
    def create_user(user_mail: str, user_name: str, user_password: str) -> Tuple[bool, Union["User", str]]:
        """
        créer un nouveau user.

        notre fonction retourne:
        - False, <message d'erreur> en cas d'erreur
        - True, User en cas de succès 
        donc, le 1er item permet de savoir si l'insertion a fonctionné 
        """
        # liste de nos erreurs
        errors = []

        # 1. on vérifie que l'utilisateur.ice a fourni toutes les données
        if user_mail is None:
            errors.append("Veuillez fournir un email")
        if user_name is None:
            errors.append("Veuillez fournir un nom d'utilisateur")
        if user_password is None:
            errors.append("Veuillez fournir un mot de passe")

        # on vérifie si il existe un autre `user` avec le même mail
        existing_user = db.session.execute(
            db.select(User).filter(User.user_mail == user_mail)
        ).scalars().all()
        if len(existing_user):
            errors.append(f"Un utilisateur existe déjà pour le mail: {user_mail}")

        # il y a eu des erreurs => pas d'insert
        if len(errors):
            return False, errors

        # pas d'erreurs => on fait l'insert
        new_user = User(
            user_name=user_name,
            user_mail=user_mail,
            user_password=generate_password_hash(user_password)
        ) 
        try:
            db.session.add(new_user)
            db.session.commit()
        except Exception as e:
            # en cas d'erreur au moment de l'insert, on retourne False et le message d'erreur de l'appli
            print(e)
            return False, str(e)

    @staticmethod
    def login(user_mail: str, user_password: str) -> Optional["User"]:
        # retourne soit un `User`, soit None 
        user = db.session.execute(
            db.select(User).filter(User.user_mail == user_mail)
        ).scalars().first()
        # on vérifie que le `User` avec ce mail a bien le bon mot de passe
        if user and check_passoword_hash(user.user_password, user_password):
            return user
        # sinon, on retourne None
        return None
```

In [ ]:
from typing import List, Optional
from pathlib import Path

from flask import Flask
from flask_sqlalchemy import SQLAlchemy
from sqlalchemy import ForeignKey
from sqlalchemy.orm import Mapped, mapped_column, relationship

# on crée appli Flask -----------------------------------------------

path_to_db = Path("./richelieu.db").absolute()
print(path_to_db)

APP_NAME = "Catalogue Richelieu"
app = Flask(APP_NAME)
app.config["SQLALCHEMY_DATABASE_URI"] = f"sqlite:///{path_to_db}"
db = SQLAlchemy(app)

# on crée notre modèle -----------------------------------------------

class IconographyUser(db.Model):
    __tablename__ = "iconography_user"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    id_iconography: Mapped[int] = mapped_column(ForeignKey("iconography.id"))
    id_user: Mapped[int] = mapped_column(ForeignKey("user.id"))


class Iconography(db.Model):
    __tablename__ = "iconography"
    
    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    title: Mapped[str]
    iiif_manifest_url: Mapped[str]
    iiif_image_url: Mapped[str]
    source_url: Mapped[Optional[str]]
    richelieu_url: Mapped[str]
    date_lower: Mapped[Optional[int]]
    date_upper: Mapped[Optional[int]]
    institution: Mapped[str]
    # NOTE: dans ce notebook, les tables `author`, `theme` et `place` n'ont pas de modèles
    #   on ne modélise donc ni `Iconography.id_author`, ni les relations `author`, `theme` et `place`

    user: Mapped[List["User"]] = relationship(
        back_populates = "iconography",
        secondary=IconographyUser.__table__
    )


class User(db.Model):
    __tablename__ = "user"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    user_name: Mapped[str]
    user_mail: Mapped[str]
    user_password: Mapped[str]
    
    iconography: Mapped[List["Iconography"]] = relationship(
        back_populates="user",
        secondary=IconographyUser.__table__
    )  

    @staticmethod
    def create_user():
        ...
    
    @staticmethod
    def login():
        ...


# on fait une requête test -----------------------------------------------

with app.app_context():
    print(db.session.execute(db.select(Iconography)).scalars().all())
